#  Organised Crime & Cybercrime in Europe
## Notebook 1: Data Loading, Validation & Merging



---

## Data Sources

This project uses two official datasets:

### Dataset 1 — Eurostat: Crime Statistics (crim_off_cat)
- **URL:** https://ec.europa.eu/eurostat/databrowser/view/crim_off_cat/default/table
- **Direct download (TSV):** https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/crim_off_cat/?format=TSV&compressed=true
- **Coverage:** EU member states + Norway, Switzerland, 2008–2022
- **Content:** Recorded criminal offences by category (homicide, robbery, drug offences, fraud, cybercrime, organised crime related)
- **How to download:**
  1. Go to https://ec.europa.eu/eurostat/databrowser/view/crim_off_cat/
  2. Click 'Download' → 'Full dataset' → choose CSV or TSV
  3. Save as `crim_off_cat.tsv` in your Google Drive or upload directly to Colab

### Dataset 2 — ENISA Threat Landscape Reports (structured extract)
- **URL:** https://www.enisa.europa.eu/topics/cyber-threats/enisa-threat-landscape
- **Alternative structured source:** European Commission Open Data Portal
  - https://data.europa.eu/data/datasets?query=cybercrime&locale=en
- **Recommended alternative (machine-readable):** Eurostat ICT security incidents (isoc_ci_in_h)
  - https://ec.europa.eu/eurostat/databrowser/view/isoc_ci_in_h/
  - Direct TSV: https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/isoc_ci_in_h/?format=TSV&compressed=true
- **Coverage:** EU countries, 2010–2022
- **Content:** Percentage of individuals/enterprises experiencing security incidents, by country and year
- **How to download:**
  1. Go to https://ec.europa.eu/eurostat/databrowser/view/isoc_ci_in_h/
  2. Download as CSV
  3. Save as `isoc_ci_in_h.csv`

---

> **Note:** The code below downloads these datasets programmatically where possible, and provides clear instructions where manual download is required.


In [17]:
# ============================================================
# CELL 1: Install & import libraries
# ============================================================
!pip install eurostat pandas numpy matplotlib seaborn plotly --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
import os
import requests
import io
import zipfile

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('✅ Libraries loaded successfully')

✅ Libraries loaded successfully


In [18]:
# ============================================================
# CELL 2: Download Dataset 1 — Eurostat Crime Statistics
# (crim_off_cat) via Eurostat REST API
# ============================================================

try:
    import eurostat
    print('Downloading crim_off_cat via eurostat package...')
    df_crime_raw = eurostat.get_data_df('crim_off_cat')
    print(f'✅ Crime dataset downloaded: {df_crime_raw.shape}')
    df_crime_raw.to_csv('crim_off_cat_raw.csv', index=False)
    print('   Saved to crim_off_cat_raw.csv')
except Exception as e:
    print(f'⚠️  eurostat package failed: {e}')
    print('   Trying direct HTTP download...')
    url = 'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/crim_off_cat/?format=SDMX-CSV&compressed=false'
    try:
        r = requests.get(url, timeout=60)
        df_crime_raw = pd.read_csv(io.StringIO(r.text))
        df_crime_raw.to_csv('crim_off_cat_raw.csv', index=False)
        print(f'✅ Crime dataset downloaded: {df_crime_raw.shape}')
    except Exception as e2:
        print(f'⚠️  HTTP download also failed: {e2}')
        print('👉 Please manually upload crim_off_cat_raw.csv to Colab (see instructions above)')


✅ Crime dataset downloaded: (1774, 21)
   Saved to crim_off_cat_raw.csv


In [19]:
# ============================================================
# CELL 3: Download Dataset 2 — Eurostat ICT Security Incidents
# (isoc_ci_in_h) — cybercrime / digital security proxy
# ============================================================

try:
    import eurostat
    print('Downloading isoc_ci_in_h via eurostat package...')
    df_cyber_raw = eurostat.get_data_df('isoc_ci_in_h')
    print(f'✅ Cyber dataset downloaded: {df_cyber_raw.shape}')
    df_cyber_raw.to_csv('isoc_ci_in_h_raw.csv', index=False)
    print('   Saved to isoc_ci_in_h_raw.csv')
except Exception as e:
    print(f'⚠️  eurostat package failed: {e}')
    url = 'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/isoc_ci_in_h/?format=SDMX-CSV&compressed=false'
    try:
        r = requests.get(url, timeout=60)
        df_cyber_raw = pd.read_csv(io.StringIO(r.text))
        df_cyber_raw.to_csv('isoc_ci_in_h_raw.csv', index=False)
        print(f'✅ Cyber dataset downloaded: {df_cyber_raw.shape}')
    except Exception as e2:
        print(f'⚠️  HTTP download also failed: {e2}')
        print('👉 Please manually upload isoc_ci_in_h_raw.csv to Colab')


✅ Cyber dataset downloaded: (730, 28)
   Saved to isoc_ci_in_h_raw.csv


In [20]:
# ============================================================
# CELL 4: Inspect raw crime dataset
# ============================================================

print('=== CRIME DATASET — FIRST LOOK ===')
print(f'Shape: {df_crime_raw.shape}')
print(f'Columns: {df_crime_raw.columns.tolist()}')
print()
display(df_crime_raw.head(10))
print()
print('--- dtypes ---')
print(df_crime_raw.dtypes)
print()
print('--- missing values ---')
print(df_crime_raw.isnull().sum())


=== CRIME DATASET — FIRST LOOK ===
Shape: (1774, 21)
Columns: ['freq', 'iccs', 'unit', 'geo\\TIME_PERIOD', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']



,freq,iccs,unit,geo\TIME_PERIOD,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,A,ICCS0101,NR,AL,88.00,82.00,118.00,124.00,125.00,107.00,98.00,54.00,71.00,52.00,51.00,58.00,52.00,61.00,44.00,35.00,41.00
1,A,ICCS0101,NR,AT,58.00,51.00,61.00,80.00,88.00,63.00,43.00,42.00,49.00,61.00,73.00,74.00,54.00,59.00,65.00,75.00,78.00
2,A,ICCS0101,NR,BA,66.00,67.00,54.00,49.00,60.00,46.00,49.00,56.00,42.00,34.00,36.00,37.00,42.00,31.00,34.00,35.00,33.00
3,A,ICCS0101,NR,BE,204.00,189.00,189.00,214.00,206.00,204.00,210.00,231.00,175.00,198.00,192.00,147.00,149.00,158.00,188.00,165.00,166.00
4,A,ICCS0101,NR,BG,172.00,150.00,148.00,128.00,141.00,109.00,112.00,126.00,79.00,95.00,92.00,81.00,67.00,89.00,76.00,74.00,79.00
5,A,ICCS0101,NR,CH,54.00,51.00,53.00,46.00,45.00,57.00,41.00,57.00,45.00,45.00,50.00,46.00,47.00,42.00,42.00,53.00,45.00
6,A,ICCS0101,NR,CY,9.00,19.00,7.00,8.00,19.00,11.00,10.00,12.00,11.00,7.00,14.00,13.00,15.00,14.00,7.00,10.00,9.00
7,A,ICCS0101,NR,CZ,113.00,105.00,105.00,83.00,95.00,90.00,81.00,88.00,65.00,40.00,55.00,81.00,57.00,46.00,79.00,68.00,69.00
8,A,ICCS0101,NR,DE,656.00,721.00,699.00,689.00,619.00,623.00,645.00,655.00,747.00,738.00,632.00,586.00,719.00,631.00,614.00,661.00,694.00
9,A,ICCS0101,NR,DK,53.00,56.00,49.00,49.00,43.00,41.00,62.00,52.00,53.00,61.00,54.00,54.00,51.00,42.00,59.00,53.00,52.00



--- dtypes ---
freq                object
iccs                object
unit                object
geo\TIME_PERIOD     object
2008               float64
2009               float64
2010               float64
2011               float64
2012               float64
2013               float64
2014               float64
2015               float64
2016               float64
2017               float64
2018               float64
2019               float64
2020               float64
2021               float64
2022               float64
2023               float64
2024               float64
dtype: object

--- missing values ---
freq                 0
iccs                 0
unit                 0
geo\TIME_PERIOD      0
2008               812
2009               786
2010               797
2011               786
2012               804
2013               798
2014               787
2015               798
2016               355
2017               346
2018               357
2019               396
2020       

In [21]:
# ============================================================
# CELL 5: Inspect raw cyber dataset
# ============================================================

print('=== CYBER DATASET — FIRST LOOK ===')
print(f'Shape: {df_cyber_raw.shape}')
print(f'Columns: {df_cyber_raw.columns.tolist()}')
print()
display(df_cyber_raw.head(10))
print()
print('--- dtypes ---')
print(df_cyber_raw.dtypes)
print()
print('--- missing values ---')
print(df_cyber_raw.isnull().sum())


=== CYBER DATASET — FIRST LOOK ===
Shape: (730, 28)
Columns: ['freq', 'unit', 'hhtyp', 'geo\\TIME_PERIOD', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']



,freq,unit,hhtyp,geo\TIME_PERIOD,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,PC_HH,A1,AL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56.73,NaN,58.56,70.76,89.59,83.19,93.15,95.59
1,A,PC_HH,A1,AT,20.08,24.53,31.01,32.67,37.60,43.30,55.32,50.66,53.97,58.48,67.89,68.60,68.41,70.08,72.48,81.14,79.03,83.54,82.55,90.39,87.67,90.50,91.49,91.59
2,A,PC_HH,A1,BA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.15,60.28,46.08,62.66,NaN,63.91,61.45,65.79
3,A,PC_HH,A1,BE,NaN,NaN,NaN,29.75,33.40,39.55,43.08,44.45,51.50,56.50,58.47,63.45,67.49,64.99,70.10,72.68,75.31,78.20,80.63,82.93,88.41,88.07,88.95,87.70
4,A,PC_HH,A1,BG,NaN,NaN,4.40,NaN,NaN,9.64,13.71,10.80,15.24,19.99,27.19,30.28,30.23,32.51,38.58,44.37,47.52,49.90,54.86,64.24,68.51,77.93,84.13,83.35
5,A,PC_HH,A1,CH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.27,NaN,NaN,86.84,NaN,92.88,NaN,97.20,NaN,99.29,NaN,97.40
6,A,PC_HH,A1,CY,NaN,NaN,47.93,15.15,16.96,17.29,25.97,30.99,23.07,38.52,37.92,46.74,48.77,51.88,54.02,61.58,74.59,75.49,86.26,86.17,89.62,85.22,87.14,95.45
7,A,PC_HH,A1,CZ,NaN,NaN,6.87,6.68,14.09,17.74,28.37,28.46,35.14,46.95,48.86,50.32,58.69,60.24,66.17,65.15,69.65,74.73,72.04,75.78,82.45,85.21,85.71,89.96
8,A,PC_HH,A1,DE,33.33,36.56,44.32,45.93,48.04,54.51,58.16,65.40,70.72,71.68,74.07,78.41,80.80,81.74,84.90,86.11,89.54,90.08,91.59,85.96,85.48,86.17,86.96,89.09
9,A,PC_HH,A1,DK,35.42,47.32,53.05,59.54,63.87,63.30,68.39,73.87,77.92,80.67,82.83,83.48,86.38,84.28,87.29,92.84,84.96,90.59,88.71,90.98,89.15,91.36,93.86,94.27



--- dtypes ---
freq                object
unit                object
hhtyp               object
geo\TIME_PERIOD     object
2002               float64
2003               float64
2004               float64
2005               float64
2006               float64
2007               float64
2008               float64
2009               float64
2010               float64
2011               float64
2012               float64
2013               float64
2014               float64
2015               float64
2016               float64
2017               float64
2018               float64
2019               float64
2020               float64
2021               float64
2022               float64
2023               float64
2024               float64
2025               float64
dtype: object

--- missing values ---
freq                 0
unit                 0
hhtyp                0
geo\TIME_PERIOD      0
2002               624
2003               592
2004               366
2005               310
2006  

In [22]:
# ============================================================
# CELL 6: Clean & normalise the CRIME dataset
# Handles both:
#   - eurostat package output (wide: year columns like '2010', '2011'...)
#   - SDMX-CSV format (long: TIME_PERIOD, OBS_VALUE columns)
# ============================================================

df_crime = df_crime_raw.copy()
df_crime.columns = [c.lower().strip() for c in df_crime.columns]

# --- Detect format ---
# Wide format (eurostat package) has numeric-looking column names for years
year_cols = [c for c in df_crime.columns if str(c).strip().isdigit() and 2000 <= int(c) <= 2030]

if year_cols:
    print(f'Detected WIDE format — {len(year_cols)} year columns found')

    # Don't assume the geo column name — find it by exclusion
    # The eurostat package puts dimension codes as non-year columns
    rename_map = {
        'iccs': 'crime_category',
        'unit': 'unit_crime',
        'freq': 'freq',
    }
    df_crime.rename(columns={k: v for k, v in rename_map.items() if k in df_crime.columns}, inplace=True)

    # The geo/country column: find it (it's whichever non-year col contains country codes)
    non_year_cols = [c for c in df_crime.columns if c not in year_cols]
    print(f'Non-year columns: {non_year_cols}')

    # Rename the geo column — it may be 'geo', 'geo\\time', or something else
    geo_col = next(
        (c for c in non_year_cols if 'geo' in str(c).lower()),
        None
    )
    if geo_col:
        df_crime.rename(columns={geo_col: 'country_code'}, inplace=True)
        print(f'Renamed geo column: "{geo_col}" → "country_code"')
    else:
        print(f'⚠️ Could not auto-detect geo column. Non-year cols: {non_year_cols}')

    # Melt year columns into long format
    id_vars = [c for c in df_crime.columns if c not in year_cols]
    df_crime = df_crime.melt(
        id_vars=id_vars,
        value_vars=year_cols,
        var_name='year',
        value_name='crime_count'
    )

# --- Common cleaning (applies to both formats after normalisation) ---
df_crime['year'] = pd.to_numeric(df_crime['year'], errors='coerce')
df_crime['crime_count'] = pd.to_numeric(df_crime['crime_count'], errors='coerce')

before = len(df_crime)
df_crime.dropna(subset=['country_code', 'year', 'crime_count'], inplace=True)
print(f'Rows dropped (missing key fields): {before - len(df_crime)}')

df_crime = df_crime[(df_crime['year'] >= 2010) & (df_crime['year'] <= 2022)]

eu_countries = [
    'AT','BE','BG','CY','CZ','DE','DK','EE','EL','ES','FI','FR',
    'HR','HU','IE','IT','LT','LU','LV','MT','NL','PL','PT','RO',
    'SE','SI','SK','NO','CH','IS','ME','MK','RS','TR'
]
df_crime = df_crime[df_crime['country_code'].isin(eu_countries)]

iccs_labels = {
    'ICCS0101': 'Intentional Homicide',
    'ICCS0102': 'Attempted Homicide',
    'ICCS0301': 'Robbery',
    'ICCS0401': 'Burglary',
    'ICCS0501': 'Theft',
    'ICCS0601': 'Fraud',
    'ICCS0602': 'Computer Fraud / Cybercrime',
    'ICCS0603': 'Identity Theft',
    'ICCS0604': 'Money Laundering',
    'ICCS0701': 'Drug Trafficking',
    'ICCS0702': 'Drug Possession',
    'ICCS1001': 'Corruption / Bribery',
    'ICCS1002': 'Embezzlement',
    'ICCS020111': 'Theft of a Motorised Land Vehicle',
    'ICCS020221': 'Burglary of Business Premises',
    'ICCS05012': 'Unlawful Acts Involving Controlled Drugs (Other)',
    'ICCS05021': 'Trafficking in Persons (Sexual Exploitation)',
    'ICCS030221': 'Embezzlement by Public Official',
    'ICCS0701': 'Drug Trafficking',
    'ICCS0703': 'Drug Possession / Use',
    'ICCS07031': 'Possession of Cannabis',
    'ICCS07041': 'Driving Under Influence of Drugs',
    'ICCS0903': 'Acts against the Natural Environment',
    'ICCS09051': 'Cybercrime Against Individuals',
    'ICCS1001': 'Corruption / Bribery',
    'ICCS1002': 'Bribery of Public Officials',
    'ICCS1003': 'Embezzlement',
    'ICCS1004': 'Trading in Influence',
}
if 'crime_category' in df_crime.columns:
    df_crime['crime_label'] = df_crime['crime_category'].map(iccs_labels).fillna(df_crime['crime_category'])

print(f'✅ Crime dataset cleaned: {df_crime.shape}')
print(f'   Years: {df_crime["year"].min():.0f} – {df_crime["year"].max():.0f}')
print(f'   Countries: {df_crime["country_code"].nunique()}')
if 'crime_category' in df_crime.columns:
    print(f'   Crime categories: {df_crime["crime_category"].nunique()}')
display(df_crime.head())


Detected WIDE format — 17 year columns found
Non-year columns: ['freq', 'crime_category', 'unit_crime', 'geo\\time_period']
Renamed geo column: "geo\time_period" → "country_code"
Rows dropped (missing key fields): 9118
✅ Crime dataset cleaned: (14272, 7)
   Years: 2010 – 2022
   Countries: 34
   Crime categories: 25


,freq,crime_category,unit_crime,country_code,year,crime_count,crime_label
3549,A,ICCS0101,NR,AT,2010,61.00,Intentional Homicide
3551,A,ICCS0101,NR,BE,2010,189.00,Intentional Homicide
3552,A,ICCS0101,NR,BG,2010,148.00,Intentional Homicide
3553,A,ICCS0101,NR,CH,2010,53.00,Intentional Homicide
3554,A,ICCS0101,NR,CY,2010,7.00,Intentional Homicide


In [23]:
# ============================================================
# CELL 7: Clean & normalise the CYBER dataset
# Expected columns: freq, unit, indic_is, hhtyp, geo, TIME_PERIOD, OBS_VALUE
# OBS_VALUE = % of individuals/households with security incidents
# ============================================================

df_cyber = df_cyber_raw.copy()
df_cyber.columns = [c.lower().strip() for c in df_cyber.columns]

year_cols = [c for c in df_cyber.columns if str(c).strip().isdigit() and 2000 <= int(c) <= 2030]

if year_cols:
    print(f'Detected WIDE format — {len(year_cols)} year columns found')

    rename_map2 = {
        'indic_is': 'cyber_indicator',
        'unit': 'unit_cyber',
        'freq': 'freq',
    }
    df_cyber.rename(columns={k: v for k, v in rename_map2.items() if k in df_cyber.columns}, inplace=True)

    non_year_cols = [c for c in df_cyber.columns if c not in year_cols]
    print(f'Non-year columns: {non_year_cols}')

    geo_col = next(
        (c for c in non_year_cols if 'geo' in str(c).lower()),
        None
    )
    if geo_col:
        df_cyber.rename(columns={geo_col: 'country_code'}, inplace=True)
        print(f'Renamed geo column: "{geo_col}" → "country_code"')
    else:
        print(f'⚠️ Could not auto-detect geo column. Non-year cols: {non_year_cols}')

    id_vars = [c for c in df_cyber.columns if c not in year_cols]
    df_cyber = df_cyber.melt(
        id_vars=id_vars,
        value_vars=year_cols,
        var_name='year',
        value_name='cyber_incident_pct'
    )

else:
    print('Detected LONG / SDMX-CSV format')
    rename_map2 = {
        'geo': 'country_code',
        'time_period': 'year',
        'obs_value': 'cyber_incident_pct',
        'indic_is': 'cyber_indicator',
        'unit': 'unit_cyber',
    }
    df_cyber.rename(columns={k: v for k, v in rename_map2.items() if k in df_cyber.columns}, inplace=True)

# --- Shared cleaning from here (unchanged) ---
df_cyber['year'] = pd.to_numeric(df_cyber['year'], errors='coerce')
df_cyber['cyber_incident_pct'] = pd.to_numeric(df_cyber['cyber_incident_pct'], errors='coerce')

before = len(df_cyber)
df_cyber.dropna(subset=['country_code', 'year', 'cyber_incident_pct'], inplace=True)
print(f'Rows dropped (missing key fields): {before - len(df_cyber)}')

df_cyber = df_cyber[(df_cyber['year'] >= 2010) & (df_cyber['year'] <= 2022)]
df_cyber = df_cyber[df_cyber['country_code'].isin(eu_countries)]

if 'cyber_indicator' in df_cyber.columns:
    print('Available cyber indicators:')
    print(df_cyber['cyber_indicator'].value_counts())

print(f'\n✅ Cyber dataset cleaned: {df_cyber.shape}')
print(f'   Years: {df_cyber["year"].min():.0f} – {df_cyber["year"].max():.0f}')
print(f'   Countries: {df_cyber["country_code"].nunique()}')
display(df_cyber.head())


Detected WIDE format — 24 year columns found
Non-year columns: ['freq', 'unit_cyber', 'hhtyp', 'geo\\time_period']
Renamed geo column: "geo\time_period" → "country_code"
Rows dropped (missing key fields): 5740

✅ Cyber dataset cleaned: (6028, 6)
   Years: 2010 – 2022
   Countries: 34


,freq,unit_cyber,hhtyp,country_code,year,cyber_incident_pct
5841,A,PC_HH,A1,AT,2010,53.97
5843,A,PC_HH,A1,BE,2010,51.50
5844,A,PC_HH,A1,BG,2010,15.24
5846,A,PC_HH,A1,CY,2010,23.07
5847,A,PC_HH,A1,CZ,2010,35.14


In [24]:
# ============================================================
# CELL 8: Aggregate cyber dataset to country-year level
# (take the mean % across all household types / indicators)
# ============================================================

df_cyber_agg = (
    df_cyber
    .groupby(['country_code', 'year'], as_index=False)
    .agg(cyber_incident_pct_mean=('cyber_incident_pct', 'mean'),
         cyber_incident_pct_max=('cyber_incident_pct', 'max'))
)

print(f'Aggregated cyber: {df_cyber_agg.shape}')
display(df_cyber_agg.head())


Aggregated cyber: (413, 4)


,country_code,year,cyber_incident_pct_mean,cyber_incident_pct_max
0,AT,2010,77.17,96.39
1,AT,2011,78.67,97.36
2,AT,2012,82.14,97.20
3,AT,2013,83.83,98.83
4,AT,2014,84.30,98.40


In [25]:
# ============================================================
# CELL 9: Aggregate crime dataset to country-year-category
# and create a wide pivot for merging
# ============================================================

# Pivot crime categories into columns
if 'crime_label' in df_crime.columns:
    df_crime_wide = df_crime.pivot_table(
        index=['country_code', 'year'],
        columns='crime_label',
        values='crime_count',
        aggfunc='sum'
    ).reset_index()
    df_crime_wide.columns.name = None
    # Flatten column names
    df_crime_wide.columns = [c if c in ['country_code', 'year'] else f'crime_{c}' for c in df_crime_wide.columns]
else:
    df_crime_wide = df_crime.groupby(['country_code', 'year'], as_index=False)['crime_count'].sum()

print(f'Wide crime dataset: {df_crime_wide.shape}')
display(df_crime_wide.head())


Wide crime dataset: (438, 27)


,country_code,year,crime_Acts against the Natural Environment,crime_Attempted Homicide,crime_Bribery of Public Officials,crime_Burglary,crime_Burglary of Business Premises,crime_Corruption / Bribery,crime_Cybercrime Against Individuals,crime_Driving Under Influence of Drugs,crime_Drug Possession / Use,crime_Drug Trafficking,crime_Embezzlement,crime_Embezzlement by Public Official,crime_Fraud,crime_ICCS03011,crime_ICCS03012,crime_ICCS0302,crime_ICCS0502,crime_Intentional Homicide,crime_Possession of Cannabis,crime_Robbery,crime_Theft,crime_Theft of a Motorised Land Vehicle,crime_Trading in Influence,crime_Trafficking in Persons (Sexual Exploitation),crime_Unlawful Acts Involving Controlled Drugs (Other)
0,AT,2010,NaN,113.34,NaN,"4,361.61",9.11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,192.95","1,235.62","2,302.24",NaN,"152,532.77",61.73,NaN,"3,537.86","89,567.77","3,649.18",NaN,"5,211.66","14,821.37"
1,AT,2011,NaN,106.25,NaN,"4,114.55",5.06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,347.70","1,329.69","2,674.56",NaN,"146,049.26",80.96,NaN,"4,004.25","85,283.27","3,943.53",NaN,"5,219.59","15,802.46"
2,AT,2012,NaN,110.30,NaN,"4,140.67",11.13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,013.67","1,270.94","2,817.11",NaN,"148,262.60",89.05,NaN,"4,088.05","86,251.76","4,077.93",NaN,"4,498.88","15,663.10"
3,AT,2013,NaN,106.24,NaN,"3,781.22",4.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"27,553.19","1,310.32","2,178.47",NaN,"155,824.11",63.75,NaN,"3,824.72","89,752.51","3,366.36",NaN,"11,613.80","16,743.79"
4,AT,2014,NaN,70.82,NaN,"3,521.92",1.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"29,506.83","1,177.68","2,143.91",NaN,"150,005.86",43.51,NaN,"3,605.90","86,491.91","3,267.97",NaN,"10,425.13","17,311.13"


In [26]:
# ============================================================
# CELL 10: MERGE both datasets on country_code + year
# ============================================================

df_merged = pd.merge(
    df_crime_wide,
    df_cyber_agg,
    on=['country_code', 'year'],
    how='inner'
)

# Add full country names
country_names = {
    'AT': 'Austria', 'BE': 'Belgium', 'BG': 'Bulgaria', 'CY': 'Cyprus',
    'CZ': 'Czech Republic', 'DE': 'Germany', 'DK': 'Denmark', 'EE': 'Estonia',
    'EL': 'Greece', 'ES': 'Spain', 'FI': 'Finland', 'FR': 'France',
    'HR': 'Croatia', 'HU': 'Hungary', 'IE': 'Ireland', 'IT': 'Italy',
    'LT': 'Lithuania', 'LU': 'Luxembourg', 'LV': 'Latvia', 'MT': 'Malta',
    'NL': 'Netherlands', 'PL': 'Poland', 'PT': 'Portugal', 'RO': 'Romania',
    'SE': 'Sweden', 'SI': 'Slovenia', 'SK': 'Slovakia', 'NO': 'Norway',
    'CH': 'Switzerland', 'IS': 'Iceland', 'ME': 'Montenegro',
    'MK': 'North Macedonia', 'RS': 'Serbia', 'TR': 'Turkey'
}
df_merged['country_name'] = df_merged['country_code'].map(country_names)

print(f'✅ MERGED DATASET: {df_merged.shape}')
print(f'   Countries: {df_merged["country_code"].nunique()}')
print(f'   Years: {df_merged["year"].min():.0f} – {df_merged["year"].max():.0f}')
print(f'   Total observations: {len(df_merged)}')
print(f'\nMissing values per column:')
print(df_merged.isnull().sum()[df_merged.isnull().sum() > 0])
display(df_merged.head())


✅ MERGED DATASET: (409, 30)
   Countries: 34
   Years: 2010 – 2022
   Total observations: 409

Missing values per column:
crime_Acts against the Natural Environment                243
crime_Attempted Homicide                                   38
crime_Bribery of Public Officials                         338
crime_Burglary of Business Premises                        72
crime_Corruption / Bribery                                327
crime_Cybercrime Against Individuals                      253
crime_Driving Under Influence of Drugs                    214
crime_Drug Possession / Use                               199
crime_Drug Trafficking                                    201
crime_Embezzlement                                        350
crime_Embezzlement by Public Official                     224
crime_Fraud                                                11
crime_ICCS03011                                            22
crime_ICCS03012                                            46
crime_ICCS

,country_code,year,crime_Acts against the Natural Environment,crime_Attempted Homicide,crime_Bribery of Public Officials,crime_Burglary,crime_Burglary of Business Premises,crime_Corruption / Bribery,crime_Cybercrime Against Individuals,crime_Driving Under Influence of Drugs,crime_Drug Possession / Use,crime_Drug Trafficking,crime_Embezzlement,crime_Embezzlement by Public Official,crime_Fraud,crime_ICCS03011,crime_ICCS03012,crime_ICCS0302,crime_ICCS0502,crime_Intentional Homicide,crime_Possession of Cannabis,crime_Robbery,crime_Theft,crime_Theft of a Motorised Land Vehicle,crime_Trading in Influence,crime_Trafficking in Persons (Sexual Exploitation),crime_Unlawful Acts Involving Controlled Drugs (Other),cyber_incident_pct_mean,cyber_incident_pct_max,country_name
0,AT,2010,NaN,113.34,NaN,"4,361.61",9.11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,192.95","1,235.62","2,302.24",NaN,"152,532.77",61.73,NaN,"3,537.86","89,567.77","3,649.18",NaN,"5,211.66","14,821.37",77.17,96.39,Austria
1,AT,2011,NaN,106.25,NaN,"4,114.55",5.06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,347.70","1,329.69","2,674.56",NaN,"146,049.26",80.96,NaN,"4,004.25","85,283.27","3,943.53",NaN,"5,219.59","15,802.46",78.67,97.36,Austria
2,AT,2012,NaN,110.30,NaN,"4,140.67",11.13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,013.67","1,270.94","2,817.11",NaN,"148,262.60",89.05,NaN,"4,088.05","86,251.76","4,077.93",NaN,"4,498.88","15,663.10",82.14,97.20,Austria
3,AT,2013,NaN,106.24,NaN,"3,781.22",4.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"27,553.19","1,310.32","2,178.47",NaN,"155,824.11",63.75,NaN,"3,824.72","89,752.51","3,366.36",NaN,"11,613.80","16,743.79",83.83,98.83,Austria
4,AT,2014,NaN,70.82,NaN,"3,521.92",1.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"29,506.83","1,177.68","2,143.91",NaN,"150,005.86",43.51,NaN,"3,605.90","86,491.91","3,267.97",NaN,"10,425.13","17,311.13",84.30,98.40,Austria


In [27]:
# ============================================================
# CELL 11: Save cleaned & merged dataset
# ============================================================

df_merged.to_csv('eu_crime_cyber_merged.csv', index=False)
df_crime.to_csv('crime_cleaned.csv', index=False)
df_cyber.to_csv('cyber_cleaned.csv', index=False)

print('✅ Files saved:')
print('   → eu_crime_cyber_merged.csv  (main analysis file)')
print('   → crime_cleaned.csv          (cleaned crime data, long format)')
print('   → cyber_cleaned.csv          (cleaned cyber data)')
print()
print('Dataset summary:')
display(df_merged.describe())


✅ Files saved:
   → eu_crime_cyber_merged.csv  (main analysis file)
   → crime_cleaned.csv          (cleaned crime data, long format)
   → cyber_cleaned.csv          (cleaned cyber data)

Dataset summary:


,year,crime_Acts against the Natural Environment,crime_Attempted Homicide,crime_Bribery of Public Officials,crime_Burglary,crime_Burglary of Business Premises,crime_Corruption / Bribery,crime_Cybercrime Against Individuals,crime_Driving Under Influence of Drugs,crime_Drug Possession / Use,crime_Drug Trafficking,crime_Embezzlement,crime_Embezzlement by Public Official,crime_Fraud,crime_ICCS03011,crime_ICCS03012,crime_ICCS0302,crime_ICCS0502,crime_Intentional Homicide,crime_Possession of Cannabis,crime_Robbery,crime_Theft,crime_Theft of a Motorised Land Vehicle,crime_Trading in Influence,crime_Trafficking in Persons (Sexual Exploitation),crime_Unlawful Acts Involving Controlled Drugs (Other),cyber_incident_pct_mean,cyber_incident_pct_max
count,409.00,166.00,371.00,71.00,409.00,337.00,82.00,156.00,195.00,210.00,208.00,59.00,185.00,398.00,387.00,363.00,180.00,402.00,405.00,191.00,383.00,358.00,393.00,74.00,386.00,349.00,409.00,409.00
mean,"2,016.09","3,848.04",381.36,451.56,"11,021.79",434.61,812.35,340.50,"1,295.19","2,657.92","82,191.02",118.10,"1,540.34","30,614.27","1,753.38","3,688.84","2,367.54","197,184.04",179.82,536.87,"5,371.52","60,613.22","18,501.60",575.95,"19,710.68","35,238.54",83.38,96.60
std,3.73,"5,528.54",647.95,"1,185.01","22,515.80","1,425.24","2,282.08",693.22,"2,950.79","4,226.52","171,842.49",150.27,"4,685.80","63,834.09","3,853.67","7,250.51","6,213.74","334,280.34",314.10,"1,332.84","10,613.78","93,090.76","54,020.21",950.88,"38,302.36","55,361.67",12.16,5.61
min,"2,010.00",1.16,0.00,0.00,54.61,0.00,0.00,0.00,2.32,0.00,6.96,0.00,2.41,17.82,3.48,0.00,0.00,371.46,1.28,0.00,18.94,45.08,62.36,0.00,16.53,267.84,38.60,55.93
25%,"2,013.00",211.80,27.91,5.06,614.22,3.15,22.26,5.34,66.87,176.87,"2,543.48",6.09,166.10,"2,883.26",128.90,137.88,281.37,"17,641.59",33.82,41.22,413.25,"6,065.93",300.50,18.32,966.64,"2,923.10",76.77,96.03
50%,"2,016.00","1,090.86",100.40,32.36,"1,973.78",20.19,82.72,53.69,272.58,955.34,"10,317.62",39.39,362.74,"6,974.05",428.01,728.30,543.37,"78,342.11",79.76,116.11,"1,548.67","22,935.57","1,686.69",191.82,"4,402.07","9,257.05",85.71,98.61
75%,"2,019.00","5,853.26",362.15,298.22,"8,893.08",174.56,503.19,492.22,913.84,"3,853.12","57,677.13",179.21,"1,262.40","28,485.56","1,557.26","2,834.02","2,085.46","199,653.13",176.55,268.61,"4,628.31","75,642.84","5,145.63",638.41,"20,193.78","41,079.43",93.35,99.84
max,"2,022.00","32,775.43","3,980.76","6,283.59","124,847.02","13,604.72","11,511.83","6,655.44","22,641.17","21,486.03","911,455.17",457.67,"42,308.77","366,192.78","38,173.98","45,301.43","54,119.94","1,466,766.79","2,325.91","9,012.67","83,675.71","464,500.36","392,410.31","4,216.18","199,108.84","256,306.99",99.33,100.00


---


Output files ready for Notebook 2 (EDA) and Notebook 3 (Analysis):
- `eu_crime_cyber_merged.csv` — main merged dataset
- `crime_cleaned.csv` — long-format crime data
- `cyber_cleaned.csv` — cyber incident data

**➡️ Continue with: `02_exploratory_data_analysis.ipynb`**
